In [1]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

Loaded .env from: /Users/shivachaitanya/Documents/work/Interactly/repos_for_shiva_learnings/workflow-sdk/.env
Added to Python path: /Users/shivachaitanya/Documents/work/Interactly/repos_for_shiva_learnings/workflow-sdk


# Quickstart — Interactly Python SDK

This notebook walks you through the core SDK features in under 5 minutes:

1. Authenticate and initialise the client
2. List existing workflows (auto-paginated)
3. Create a workflow from a typed config
4. Execute a run over HTTP and inspect the events
5. Fetch the completed run
6. Clean up

**Prerequisites:** Set the following environment variables (or pass them to the client constructor):

```bash
export INTERACTLY_API_KEY="your-api-key"
export INTERACTLY_TEAM_ID="your-team-id"
export INTERACTLY_USER_ID="your-user-id"
export INTERACTLY_BASE_URL="https://your-instance.interactly.io"
```


> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [2]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

TEAM ID is: 67458e762b7d3dc15aaea5b5
USER ID is: 687b1a4f745c8e6806c98d91
BASE URL is: https://api-dev.interactly.ai/workflows


## Initialize Workflow Client Handle

In [3]:
import os
from interactly import AsyncWorkflowClient, WorkflowCommand

# Credentials are read from environment variables automatically.
# You can also pass them explicitly:
#   client = AsyncWorkflowClient(
#       api_key="sk-…",
#       team_id="team-…",
#       user_id="user-…",
#       base_url="https://your-instance.interactly.io",
#   )
client = AsyncWorkflowClient()
print("Client initialised. Base URL:", client._base_url)

Client initialised. Base URL: https://api-dev.interactly.ai/workflows


## 1. List existing workflows (auto-paginate)

In [4]:
from typing import List
from interactly.types.workflows.workflow import Workflow
from interactly._pagination import AsyncPage


page: AsyncPage[Workflow] = await client.workflows.list(size=25)
all_wfs: List[Workflow] = await page.list_all()   # transparently fetches additional pages

print(f"Total workflows in team: {len(all_wfs)}")
for wf in all_wfs[:5]:
    print(f"  {wf.id}  {wf.name!r}")

Total workflows in team: 8
  6a47f332edea7a0565d94e74  'Test ST Protocol Node'
  6a404f1fa87b93a326d8aa09  'Attachable llm test'
  6a26d694b62b484e9ab8b980  '[Guardrail N-Strike]:  Branching ChatBot with more flavors'
  6a17af9788ac7012f13964ac  'Clinical Triage Workflow (Adult Telephone Triage)'
  6a01873b79831bec0ce96871  'Warm Transfer Conversational Workflow'


## 2. Create a new workflow

In [5]:
from interactly.configs import (
    DirectEdgeConfig,
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)
from interactly.types.workflows.workflow import Workflow

# A minimal *runnable* workflow: greet the caller, then say goodbye.
# (An empty workflow can be created, but it can't be executed — there is no
# start node for the runtime to enter.)
greeting = SayStaticMessageNodeConfig(
    name="Greeting",
    is_start=True,
    static_messages_config=StaticMessagesConfig(static_messages=["Hi! Welcome to Interactly."]),
)
goodbye = SayStaticMessageNodeConfig(
    name="Goodbye",
    static_messages_config=StaticMessagesConfig(static_messages=["Goodbye!"]),
)
config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(name="Quickstart Notebook Workflow", category="SDK Examples"),
    node_configs=[greeting, goodbye],
    edge_configs=[
        DirectEdgeConfig(
            source_node_logical_id=greeting.logical_id,
            destination_node_logical_id=goodbye.logical_id,
            name="greeting->goodbye",
        )
    ],
)

workflow: Workflow = await client.workflows.create_from_config(config)

print(f"Created workflow  id={workflow.id}  name={workflow.name!r}")
WORKFLOW_ID = workflow.id

Created workflow  id=6a568999d462c1a6e2efaa09  name='Quickstart Notebook Workflow'


## 3. Execute a run

The `client.runs.execute()` method triggers a workflow run over a standard HTTP REST API.
It waits for the turn to complete and returns an `InteractiveRunResponse` whose `events`
are raw dicts. The two cells below first print those raw event dicts, then use
`parse_event()` to turn each one into its strongly-typed event model.


In [6]:
from interactly.types.runs.interactive_run import InteractiveRunResponse

print("Starting run (HTTP execute)…")

response: InteractiveRunResponse = await client.runs.execute(
    workflow_id=WORKFLOW_ID,
    command=WorkflowCommand.START,
)

print(f"Run ended — status: {response.status}")
print(f"Events received: {len(response.events)}")
RUN_ID = response.run_id

Starting run (HTTP execute)…
Run ended — status: completed
Events received: 15


In [7]:
# List the events for this run, in chronological order as dicts
for event in response.events:
    print(f"  {event}")

  {'logical_id': 'event_5eb2486f-9df4-46ab-9fe8-91a6afac1ee8', 'timestamp': 1784056217.988359, 'origin_thread_id': '0', 'comments': [], 'interrupted': False, 'origin_node_logical_id': 'node_92cc5950-8f21-4c2e-9983-41c1d58fcf9e', 'origin_node_name': 'Greeting', 'origin_node_run_id': '1', 'type': 'start_node_run', 'run_input': None}
  {'logical_id': 'event_c31f5295-1934-4373-bb2e-1ec242785cff', 'timestamp': 1784056217.988569, 'origin_thread_id': '0', 'comments': [], 'interrupted': False, 'origin_node_logical_id': 'node_92cc5950-8f21-4c2e-9983-41c1d58fcf9e', 'origin_node_name': 'Greeting', 'origin_node_run_id': '1', 'type': 'say_static', 'message': {'content': 'Hi! Welcome to Interactly.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'ai', 'name': None, 'id': 'assistant_message_b0e32b12-c11e-47da-ae2f-f7546c8b8bed', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None}}
  {'logical_id': 'assistant_message_b0e32b12-c11e-47da-ae2f-f7546c8b8bed', 'timestamp': 17840

In [8]:
from interactly.runtime.events import parse_event

# Parse the raw dict events into their strongly-typed representations
typed_events = [parse_event(e) for e in response.events]

for typed_event in typed_events:
    print(f"  {type(typed_event).__name__}: {typed_event.model_dump_json(indent=2)}")

  StartRunNodeEvent: {
  "logical_id": "event_5eb2486f-9df4-46ab-9fe8-91a6afac1ee8",
  "timestamp": 1784056217.988359,
  "origin_thread_id": "0",
  "comments": [],
  "interrupted": false,
  "origin_node_logical_id": "node_92cc5950-8f21-4c2e-9983-41c1d58fcf9e",
  "origin_node_name": "Greeting",
  "origin_node_run_id": "1",
  "type": "start_node_run",
  "run_input": null
}
  SayStaticMessageNodeEvent: {
  "logical_id": "event_c31f5295-1934-4373-bb2e-1ec242785cff",
  "timestamp": 1784056217.988569,
  "origin_thread_id": "0",
  "comments": [],
  "interrupted": false,
  "origin_node_logical_id": "node_92cc5950-8f21-4c2e-9983-41c1d58fcf9e",
  "origin_node_name": "Greeting",
  "origin_node_run_id": "1",
  "type": "say_static",
  "message": {
    "content": "Hi! Welcome to Interactly.",
    "additional_kwargs": {},
    "response_metadata": {},
    "type": "ai",
    "name": null,
    "id": "assistant_message_b0e32b12-c11e-47da-ae2f-f7546c8b8bed",
    "tool_calls": [],
    "invalid_tool_calls": 

## 4. Fetch the completed run

In [9]:
from interactly import Run

run: Run = await client.runs.get(RUN_ID)
print(f"Run {run.id}  status={run.status}  started={run.started_at}")

Run 6a5689990c61904bd462c1ba  status=completed  started=2026-07-14 19:10:17.855000+00:00


## 5. Cleanup

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
print(f"Workflow {WORKFLOW_ID} deleted.")

await client.close()